# Adversarial Threats to Cloud IDS — ROCm Pipeline (CUDA device)
Code-only. All hyperparameters in `CONFIG` (Cell 2). Env vars `RUN_SEED` / `RUN_DATA_SEED` override seeds for multi-run automation.


In [1]:
!pip install -q adversarial-robustness-toolbox shap imbalanced-learn tqdm pandas scikit-learn matplotlib



[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ===== SECTION 1: imports, seeds, CONFIG =====
import os
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "11.0.0"  # must precede `import torch`

import glob, time, json, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif, VarianceThreshold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from imblearn.over_sampling import SMOTE
import shap
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent, DeepFool
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# DATA_SEED (fixed): train/test split, SMOTE, feature selection, Square Attack RNG.
# SEED (varies per run): weight init + training stochasticity only.
DATA_SEED = 42
SEED = 42

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
# Nudge intra-backend determinism (does NOT guarantee cross-backend bitwise match -
# confirmed via layer-hook diagnostic that Dropout's device-level RNG diverges
# between DirectML and ROCm even with identical torch.manual_seed). warn_only=True
# so unsupported ROCm/MIOpen ops fall back instead of hard-crashing.
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as e:
    print(f"determinism flag not fully supported: {e}")


class Logger:
    def __init__(self):
        self.t0 = time.time()
        self.n = 0

    def _t(self):
        s = time.time() - self.t0
        m, s = divmod(s, 60)
        h, m = divmod(m, 60)
        return f"{int(h):02d}:{int(m):02d}:{s:05.2f}"

    def section(self, title):
        self.n += 1
        print(f"\n{'='*78}\n>>> [{self._t()}] SECTION {self.n}: {title}\n{'='*78}")

    def info(self, msg): print(f"    . [{self._t()}] {msg}")
    def ok(self, msg): print(f"    v [{self._t()}] {msg}")
    def warn(self, msg): print(f"    ! [{self._t()}] WARNING: {msg}")
    def metric(self, label, value): print(f"    > [{self._t()}] {label:<42s} {value}")


log = Logger()

OUTPUT_DIR = f"./output/seed_{SEED}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/checkpoints", exist_ok=True)

CONFIG = {
    # data
    "day_patterns": None,
    "max_rows_per_file": None,
    "test_size": 0.2,

    # feature selection
    "top_t_mutual_info": 25,
    "top_t_shap": 15,
    "mi_var_thresh": 1e-5,
    "mi_sample_size": 50000,
    "shap_background_samples": 100,

    # architecture / training
    "hidden_sizes": [128, 64, 32],
    "dropout": 0.3,
    "learning_rate": 0.001,
    "val_split": 0.1,
    "epochs_baseline": 50,
    "epochs_defended_stage1": 50,
    "epochs_defended_stage2": 15,
    "batch_size": 512,

    # attacks
    "attack_eps": 0.05,
    "pgd_iters": 20,
    "pgd_eps_step_ratio": 0.1,
    "deepfool_iters": 50,
    "square_iters": 300,
    "square_p_init": 0.3,
    "n_attack_eval_samples": 2000,
    "n_square_samples": 1000,
    "n_shap_samples": 300,
    "n_adv_train_samples": 5000,

    # eps sweep (optional post-hoc robustness curve)
    "run_eps_sweep": True,
    "eps_sweep_values": [0.02, 0.05, 0.08, 0.10, 0.12],

    # seeds
    "seed": SEED,
    "data_seed": DATA_SEED,
}

print(f"Run started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | SEED={SEED} | DATA_SEED={DATA_SEED}")
log.info(f"OUTPUT_DIR={OUTPUT_DIR}")
log.info(f"CONFIG={json.dumps(CONFIG, indent=2)}")


def to_onehot(y, n_classes=2):
    return np.eye(n_classes, dtype=np.float32)[y]


F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Run started: 2026-08-02 20:01:45 | SEED=42 | DATA_SEED=42
    . [00:00:00.00] OUTPUT_DIR=./output/seed_42
    . [00:00:00.00] CONFIG={
  "day_patterns": null,
  "max_rows_per_file": null,
  "test_size": 0.2,
  "top_t_mutual_info": 25,
  "top_t_shap": 15,
  "mi_var_thresh": 1e-05,
  "mi_sample_size": 50000,
  "shap_background_samples": 100,
  "hidden_sizes": [
    128,
    64,
    32
  ],
  "dropout": 0.3,
  "learning_rate": 0.001,
  "val_split": 0.1,
  "epochs_baseline": 50,
  "epochs_defended_stage1": 50,
  "epochs_defended_stage2": 15,
  "batch_size": 512,
  "attack_eps": 0.05,
  "pgd_iters": 20,
  "pgd_eps_step_ratio": 0.1,
  "deepfool_iters": 50,
  "square_iters": 300,
  "square_p_init": 0.3,
  "n_attack_eval_samples": 2000,
  "n_square_samples": 1000,
  "n_shap_samples": 300,
  "n_adv_train_samples": 5000,
  "run_eps_sweep": true,
  "eps_sweep_values": [
    0.02,
    0.05,
    0.08,
    0.1,
    0.12
  ],
  "seed": 42,
  "data_seed": 42
}


In [3]:
# ===== SECTION 2: device setup (ROCm) =====
log.section("Device setup")

CPU_DEVICE = torch.device("cpu")
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    log.ok(f"GPU acquired: {DEVICE} ({torch.cuda.get_device_name(0)})")
else:
    log.warn("CUDA/ROCm not available - falling back to CPU. "
             "Check HSA_OVERRIDE_GFX_VERSION was set before `import torch` and the kernel uses rocm_env.")
    DEVICE = CPU_DEVICE

log.info(f"torch={torch.__version__} | train/infer device={DEVICE} | ART attack device={CPU_DEVICE} (fixed, CPU)")



>>> [00:00:00.07] SECTION 1: Device setup
    v [00:00:00.27] GPU acquired: cuda (AMD Radeon RX 7600)
    . [00:00:00.27] torch=2.9.1+rocm7.2.1 | train/infer device=cuda | ART attack device=cpu (fixed, CPU)


In [4]:
# ===== SECTION 3: data discovery =====
log.section("Data discovery")

DATA_DIR = "data"
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"'{DATA_DIR}/' not found. Place the CSE-CIC-IDS2018 parquet/csv files there.")

all_files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.parquet"), recursive=True))
if not all_files:
    all_files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
if not all_files:
    raise FileNotFoundError(f"No .parquet or .csv files found under '{DATA_DIR}/'.")

log.ok(f"Found {len(all_files)} file(s):")
for p in all_files:
    log.info(f"{os.path.basename(p)} ({os.path.getsize(p)/1e6:,.1f} MB)")



>>> [00:00:00.29] SECTION 2: Data discovery
    v [00:00:00.29] Found 10 file(s):
    . [00:00:00.29] Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet (83.5 MB)
    . [00:00:00.30] Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter.parquet (76.4 MB)
    . [00:00:00.30] DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet (89.5 MB)
    . [00:00:00.30] DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlowMeter.parquet (33.4 MB)
    . [00:00:00.30] DoS1-Thursday-15-02-2018_TrafficForML_CICFlowMeter.parquet (98.6 MB)
    . [00:00:00.30] DoS2-Friday-16-02-2018_TrafficForML_CICFlowMeter.parquet (58.0 MB)
    . [00:00:00.30] Infil1-Wednesday-28-02-2018_TrafficForML_CICFlowMeter.parquet (51.0 MB)
    . [00:00:00.30] Infil2-Thursday-01-03-2018_TrafficForML_CICFlowMeter.parquet (29.3 MB)
    . [00:00:00.30] Web1-Thursday-22-02-2018_TrafficForML_CICFlowMeter.parquet (104.0 MB)
    . [00:00:00.30] Web2-Friday-23-02-2018_TrafficForML_CICFlowMeter.parquet (101.4 MB)


In [5]:
# ===== SECTION 4: data loading =====
log.section("Data loading")

DAY_PATTERNS = CONFIG["day_patterns"]
if DAY_PATTERNS is None:
    selected_files = all_files
else:
    selected_files = [p for p in all_files if any(pat in os.path.basename(p) for pat in DAY_PATTERNS)]
    for pat in DAY_PATTERNS:
        if not any(pat in os.path.basename(p) for p in all_files):
            log.warn(f"No file matches pattern '{pat}'")
if not selected_files:
    log.warn("No file matched day_patterns - using all found files.")
    selected_files = all_files

MAX_ROWS_PER_FILE = CONFIG["max_rows_per_file"]

dfs = []
for p in tqdm(selected_files, desc="Loading"):
    d = pd.read_parquet(p, engine="pyarrow") if p.endswith(".parquet") else pd.read_csv(p, low_memory=False)
    float_cols = d.select_dtypes(include=["float64"]).columns
    d[float_cols] = d[float_cols].astype(np.float32)
    if MAX_ROWS_PER_FILE is not None and len(d) > MAX_ROWS_PER_FILE:
        d = d.sample(n=MAX_ROWS_PER_FILE, random_state=DATA_SEED)
    log.info(f"{os.path.basename(p)}: {d.shape[0]:,} rows")
    dfs.append(d)

df = pd.concat(dfs, ignore_index=True)
del dfs
log.ok(f"Combined raw shape: {df.shape[0]:,} rows x {df.shape[1]} cols")



>>> [00:00:00.32] SECTION 3: Data loading


Loading:  10%|█         | 1/10 [00:02<00:19,  2.22s/it]

    . [00:00:02.54] Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet: 771,587 rows


Loading:  20%|██        | 2/10 [00:02<00:10,  1.36s/it]

    . [00:00:03.31] Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter.parquet: 619,346 rows


...

Loading:  90%|█████████ | 9/10 [00:07<00:00,  1.48it/s]

    . [00:00:07.70] Web1-Thursday-22-02-2018_TrafficForML_CICFlowMeter.parquet: 830,224 rows


Loading: 100%|██████████| 10/10 [00:08<00:00,  1.20it/s]

    . [00:00:08.63] Web2-Friday-23-02-2018_TrafficForML_CICFlowMeter.parquet: 829,405 rows


    v [00:00:09.77] Combined raw shape: 6,659,532 rows x 78 cols


In [6]:
# ===== SECTION 5: data cleaning =====
log.section("Data cleaning")

df.columns = df.columns.str.strip()

n_before = df.shape[1]
df = df.loc[:, ~df.columns.duplicated()]
if df.shape[1] != n_before:
    log.warn(f"Dropped {n_before - df.shape[1]} duplicate-named column(s)")

cat_cols = df.select_dtypes(include="category").columns
if len(cat_cols) > 0:
    df[cat_cols] = df[cat_cols].astype(object)
    log.info(f"Cast {len(cat_cols)} category column(s) to object")

label_col = next((c for c in df.columns if c.strip().lower() == "label"), None)
if label_col is None:
    raise KeyError(f"No 'Label' column found. Columns: {list(df.columns)}")
df = df.rename(columns={label_col: "Label"})

n_before = len(df)
df = df[df["Label"].astype(str).str.strip().str.lower() != "label"]
if n_before != len(df):
    log.warn(f"Dropped {n_before - len(df):,} embedded-header rows")

drop_cols = [c for c in ["Timestamp", "Flow ID", "Src IP", "Dst IP", "Source IP",
                          "Destination IP", "Src IP Addr", "Dst IP Addr"] if c in df.columns]
if drop_cols:
    df = df.drop(columns=drop_cols)
    log.info(f"Dropped identity/temporal columns: {drop_cols}")

n_before = len(df)
df = df.drop_duplicates()
log.ok(f"Dropped {n_before - len(df):,} duplicate rows - {len(df):,} rows remain")



>>> [00:00:09.86] SECTION 4: Data cleaning
    v [00:00:49.49] Dropped 573,667 duplicate rows - 6,085,865 rows remain


In [7]:
# ===== SECTION 6: binary labeling + encoding =====
log.section("Binary labeling and encoding")

df["y"] = (df["Label"].astype(str).str.strip().str.lower() != "benign").astype(int)
df = df.drop(columns=["Label"])
log.ok(f"Binary label: {np.bincount(df['y'])} (0=benign, 1=attack)")

proto_col = next((c for c in df.columns if c.strip().lower() == "protocol"), None)
structural_features = []
if proto_col is not None:
    pre_cols = set(df.columns)
    df = pd.get_dummies(df, columns=[proto_col], prefix="proto")
    structural_features = sorted(set(df.columns) - pre_cols)  # auto-detected, not hardcoded
    log.info(f"One-hot encoded '{proto_col}' -> structural_features={structural_features}")
CONFIG["structural_features"] = structural_features

non_numeric = df.select_dtypes(include=["object", "category"]).columns
for c in non_numeric:
    df[c] = pd.to_numeric(df[c], errors="coerce")
if len(non_numeric) > 0:
    log.info(f"Coerced {len(non_numeric)} non-numeric column(s) to numeric")
df = df.dropna()

X = df.drop(columns=["y"]).reset_index(drop=True).astype(np.float32)
y = df["y"].values.astype(np.int64)
log.ok(f"Final feature matrix: {X.shape} | class balance: {np.bincount(y)}")



>>> [00:00:49.52] SECTION 5: Binary labeling and encoding
    v [00:00:49.91] Binary label: [4760022 1325843] (0=benign, 1=attack)
    . [00:00:50.01] One-hot encoded 'Protocol' -> structural_features=['proto_0', 'proto_17', 'proto_6']
    v [00:00:50.75] Final feature matrix: (6085865, 79) | class balance: [4760022 1325843]


In [8]:
# ===== SECTION 7: mutual-information feature selection =====
log.section("Mutual-information feature selection")

def robust_feature_selection(X, y, top_t, var_thresh=None, mi_sample_size=None, seed=DATA_SEED):
    var_thresh = CONFIG["mi_var_thresh"] if var_thresh is None else var_thresh
    mi_sample_size = CONFIG["mi_sample_size"] if mi_sample_size is None else mi_sample_size

    vt = VarianceThreshold(threshold=var_thresh)
    vt.fit(X)
    kept = X.columns[vt.get_support()]
    log.info(f"Variance threshold kept {len(kept)}/{X.shape[1]} features")
    X_v = X[kept]

    rng = np.random.RandomState(seed)
    if len(X_v) > mi_sample_size:
        idx = rng.choice(len(X_v), mi_sample_size, replace=False)
        X_mi, y_mi = X_v.iloc[idx], y[idx]
    else:
        X_mi, y_mi = X_v, y

    mi = mutual_info_classif(X_mi, y_mi, random_state=seed)
    mi_series = pd.Series(mi, index=X_v.columns).sort_values(ascending=False)
    return mi_series.head(top_t).index.tolist(), mi_series

selected_features, mi_scores = robust_feature_selection(X, y, top_t=CONFIG["top_t_mutual_info"])
log.ok(f"Selected {len(selected_features)} features: {selected_features}")



>>> [00:00:50.79] SECTION 6: Mutual-information feature selection
    . [00:00:56.61] Variance threshold kept 71/79 features
    v [00:01:09.00] Selected 25 features: ['Bwd Packets Length Total', 'Avg Bwd Segment Size', 'Subflow Bwd Bytes', 'Bwd Packet Length Mean', 'Fwd Packets Length Total', 'Subflow Fwd Bytes', 'Bwd Packet Length Max', 'Fwd Packet Length Max', 'Avg Fwd Segment Size', 'Fwd Packet Length Mean', 'Bwd Packet Length Std', 'Fwd Packet Length Std', 'Packet Length Variance', 'Packet Length Std', 'Packet Length Mean', 'Packet Length Max', 'Avg Packet Size', 'Fwd Header Length', 'Init Fwd Win Bytes', 'Fwd IAT Max', 'Fwd IAT Total', 'Init Bwd Win Bytes', 'Flow IAT Max', 'Bwd Header Length', 'Fwd IAT Mean']


In [9]:
# ===== SECTION 8: split / scale / balance =====
log.section("Split / scale / balance")

X_sel = X[selected_features].values
X_train, X_test, y_train, y_test = train_test_split(
    X_sel, y, test_size=CONFIG["test_size"], random_state=DATA_SEED, stratify=y
)
log.info(f"Train={X_train.shape} Test={X_test.shape}")

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

sm = SMOTE(random_state=DATA_SEED)
X_train_bal, y_train_bal = sm.fit_resample(X_train, y_train)
X_train_bal = X_train_bal.astype(np.float32)
log.ok(f"SMOTE-balanced: {X_train_bal.shape} | balance={np.bincount(y_train_bal)}")



>>> [00:01:09.04] SECTION 7: Split / scale / balance
    . [00:01:13.24] Train=(4868692, 25) Test=(1217173, 25)
    v [00:13:22.78] SMOTE-balanced: (7616036, 25) | balance=[3808018 3808018]


In [10]:
# ===== SECTION 9: model + training/eval/attack helpers =====

class IDSNet(nn.Module):
    def __init__(self, input_dim, hidden_sizes=None, dropout=None):
        super().__init__()
        hidden_sizes = CONFIG["hidden_sizes"] if hidden_sizes is None else hidden_sizes
        dropout = CONFIG["dropout"] if dropout is None else dropout
        layers, prev = [], input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_model(model, X_train, y_train, epochs, batch_size, device,
                 desc="Training", val_split=None, seed=DATA_SEED, lr=None):
    val_split = CONFIG["val_split"] if val_split is None else val_split
    lr = CONFIG["learning_rate"] if lr is None else lr

    model.to(device)
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(X_train))
    n_val = max(1, int(len(X_train) * val_split))
    val_idx, tr_idx = idx[:n_val], idx[n_val:]

    train_ds = torch.utils.data.TensorDataset(
        torch.tensor(X_train[tr_idx], dtype=torch.float32),
        torch.tensor(y_train[tr_idx], dtype=torch.long),
    )
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    X_val_t = torch.tensor(X_train[val_idx], dtype=torch.float32).to(device)
    y_val_t = torch.tensor(y_train[val_idx], dtype=torch.long).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"loss": [], "val_loss": [], "val_acc": []}
    pbar = tqdm(range(epochs), desc=desc)
    for _ in pbar:
        model.train()
        epoch_loss, n_batches = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
        model.eval()
        with torch.no_grad():
            val_out = model(X_val_t)
            val_loss = criterion(val_out, y_val_t).item()
            val_acc = (val_out.argmax(1) == y_val_t).float().mean().item()
        history["loss"].append(epoch_loss / max(n_batches, 1))
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        pbar.set_postfix(loss=f"{epoch_loss/max(n_batches,1):.4f}", val_acc=f"{val_acc:.4f}")
    return history


def predict_proba_numpy(model, X, device, batch_size=4096):
    model.eval()
    model.to(device)
    outs = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.tensor(np.asarray(X[i:i + batch_size]), dtype=torch.float32).to(device)
            outs.append(torch.softmax(model(xb), dim=1).cpu().numpy())
    return np.vstack(outs)


def evaluate(model, X, y, device, label=""):
    t0 = time.time()
    probs = predict_proba_numpy(model, X, device)
    infer_ms = (time.time() - t0) / len(X) * 1000
    preds = probs.argmax(axis=1)
    m = dict(
        accuracy=accuracy_score(y, preds),
        f1=f1_score(y, preds, zero_division=0),
        precision=precision_score(y, preds, zero_division=0),
        recall=recall_score(y, preds, zero_division=0),
        infer_ms=infer_ms,
    )
    if label:
        log.metric(label, f"Acc={m['accuracy']:.3f} F1={m['f1']:.3f} "
                           f"Prec={m['precision']:.3f} Rec={m['recall']:.3f} Infer={infer_ms:.4f}ms")
    return m


def attack_diagnostics(model, X_clean, X_adv, y_true, device):
    l_inf = np.abs(X_adv - X_clean).max(axis=1)
    l2 = np.linalg.norm(X_adv - X_clean, axis=1)
    clean_preds = predict_proba_numpy(model, X_clean, device).argmax(1)
    adv_preds = predict_proba_numpy(model, X_adv, device).argmax(1)
    was_correct = (clean_preds == y_true)
    n_correct = int(was_correct.sum())
    success_rate = float(((clean_preds != adv_preds) & was_correct).sum() / max(n_correct, 1))
    return dict(mean_linf=float(l_inf.mean()), max_linf=float(l_inf.max()),
                mean_l2=float(l2.mean()), success_rate=success_rate, n_originally_correct=n_correct)


def build_mutable_mask(feature_names, structural_names):
    return np.array([f not in structural_names for f in feature_names])


def apply_feature_mask(X_adv, X_clean, mask):
    X_masked = X_clean.copy()
    X_masked[:, mask] = X_adv[:, mask]
    return X_masked


def make_art_classifier(model, input_dim, clip_values):
    cpu_model = IDSNet(input_dim)
    cpu_model.load_state_dict(model.state_dict())
    cpu_model.to("cpu")
    cpu_model.eval()
    optimizer = torch.optim.Adam(cpu_model.parameters(), lr=CONFIG["learning_rate"])
    clf = PyTorchClassifier(
        model=cpu_model, loss=nn.CrossEntropyLoss(), optimizer=optimizer,
        input_shape=(input_dim,), nb_classes=2, clip_values=clip_values, device_type="cpu",
    )
    return clf, cpu_model


def cap_to_eps(X_adv, X_clean, eps, clip_values):
    delta = np.clip(X_adv - X_clean, -eps, eps)
    return np.clip(X_clean + delta, clip_values[0], clip_values[1]).astype(np.float32)


def make_attacks(clf, eps):
    return {
        "FGSM": FastGradientMethod(estimator=clf, eps=eps),
        "PGD": ProjectedGradientDescent(estimator=clf, eps=eps,
                                         eps_step=eps * CONFIG["pgd_eps_step_ratio"],
                                         max_iter=CONFIG["pgd_iters"]),
        "DeepFool": DeepFool(classifier=clf, max_iter=CONFIG["deepfool_iters"]),
    }


def tabular_square_attack(model, X, y, device, eps, n_iter=None, p_init=None, seed=DATA_SEED, desc="Square Attack"):
    n_iter = CONFIG["square_iters"] if n_iter is None else n_iter
    p_init = CONFIG["square_p_init"] if p_init is None else p_init
    rng = np.random.RandomState(seed)
    n_samples, n_features = X.shape
    X_adv = X.copy()
    lo, hi = float(X.min()), float(X.max())

    def predict_labels(Xb):
        return predict_proba_numpy(model, Xb, device).argmax(axis=1)

    still_correct = predict_labels(X_adv) == y
    n_originally_correct = int(still_correct.sum())
    pbar = tqdm(range(n_iter), desc=desc)
    for it in pbar:
        n_remaining = int(still_correct.sum())
        pbar.set_postfix(remaining=n_remaining, total=n_samples)
        if n_remaining == 0:
            break
        p = max(p_init * (1 - it / n_iter), 0.05)
        n_perturb = max(1, int(p * n_features))
        active_idx = np.where(still_correct)[0]
        candidate = X_adv[active_idx].copy()
        for row in range(len(active_idx)):
            feat_idx = rng.choice(n_features, n_perturb, replace=False)
            delta = rng.choice([-eps, eps], size=n_perturb)
            candidate[row, feat_idx] = np.clip(candidate[row, feat_idx] + delta, lo, hi)
        preds = predict_labels(candidate)
        flipped = preds != y[active_idx]
        X_adv[active_idx[flipped]] = candidate[flipped]
        still_correct[active_idx[flipped]] = False
    pbar.close()

    success_rate = 1 - (int(still_correct.sum()) / max(n_originally_correct, 1))
    return X_adv, success_rate

log.ok("Model + helper functions defined")


    v [00:13:22.86] Model + helper functions defined


In [11]:
# ===== SECTION 10: baseline model training =====
log.section("Baseline model training")

input_dim = X_train_bal.shape[1]
baseline_model = IDSNet(input_dim)
log.info(f"input_dim={input_dim} hidden_sizes={CONFIG['hidden_sizes']} epochs={CONFIG['epochs_baseline']} device={DEVICE}")

t0 = time.time()
train_model(baseline_model, X_train_bal, y_train_bal, epochs=CONFIG["epochs_baseline"],
            batch_size=CONFIG["batch_size"], device=DEVICE, desc="Baseline")
baseline_train_time = time.time() - t0
log.ok(f"Baseline trained in {baseline_train_time/60:.2f} min")

torch.save(baseline_model.state_dict(), f"{OUTPUT_DIR}/checkpoints/baseline_seed{SEED}.pt")

baseline_clean_metrics = evaluate(baseline_model, X_test, y_test, DEVICE, "Baseline (clean)")



>>> [00:13:22.88] SECTION 8: Baseline model training
    . [00:13:23.24] input_dim=25 hidden_sizes=[128, 64, 32] epochs=50 device=cuda


Baseline: 100%|██████████| 50/50 [1:11:16<00:00, 85.52s/it, loss=0.1554, val_acc=0.9429]


    v [01:24:46.14] Baseline trained in 71.38 min
    > [01:24:47.10] Baseline (clean)                           Acc=0.954 F1=0.897 Prec=0.873 Rec=0.923 Infer=0.0004ms


In [12]:
# ===== SECTION 11: FGSM/PGD/DeepFool vs baseline (CPU via ART) =====
log.section("Adversarial attacks vs baseline")

EPS = CONFIG["attack_eps"]
clip_values = (float(X_train_bal.min()), float(X_train_bal.max()))
classifier, _ = make_art_classifier(baseline_model, input_dim, clip_values)

mutable_mask = build_mutable_mask(selected_features, CONFIG["structural_features"])
log.info(f"Mutable features: {mutable_mask.sum()}/{len(mutable_mask)}")

attacks = make_attacks(classifier, EPS)

N_ATTACK = min(CONFIG["n_attack_eval_samples"], len(X_test))
idx = np.random.RandomState(SEED).choice(len(X_test), N_ATTACK, replace=False)
X_atk, y_atk = X_test[idx], y_test[idx]

adv_results_before = {}
X_adv_store = {}
for name, atk in tqdm(list(attacks.items()), desc="Attacks vs baseline"):
    X_adv = atk.generate(x=X_atk, y=to_onehot(y_atk))
    X_adv = apply_feature_mask(X_adv, X_atk, mutable_mask)
    if name == "DeepFool":
        X_adv = cap_to_eps(X_adv, X_atk, EPS, clip_values)
    X_adv_store[name] = X_adv
    adv_results_before[name] = evaluate(baseline_model, X_adv, y_atk, DEVICE, f"Baseline under {name}")
    diag = attack_diagnostics(baseline_model, X_atk, X_adv, y_atk, DEVICE)
    adv_results_before[name].update(diag)
    log.metric(f"{name} success rate / L_inf", f"{diag['success_rate']*100:.1f}% / {diag['mean_linf']:.4f}")



>>> [01:24:47.16] SECTION 9: Adversarial attacks vs baseline
    . [01:24:47.25] Mutable features: 25/25


Attacks vs baseline:  33%|███▎      | 1/3 [00:00<00:00,  2.32it/s]

    > [01:24:47.70] Baseline under FGSM                        Acc=0.750 F1=0.260 Prec=0.346 Rec=0.208 Infer=0.0015ms
    > [01:24:47.71] FGSM success rate / L_inf                  21.6% / 0.0499



Attacks vs baseline:  67%|██████▋   | 2/3 [00:03<00:02,  2.12s/it]

    > [01:24:51.00] Baseline under PGD                         Acc=0.686 F1=0.176 Prec=0.198 Rec=0.158 Infer=0.0018ms
    > [01:24:51.00] PGD success rate / L_inf                   27.9% / 0.0462



Attacks vs baseline: 100%|██████████| 3/3 [00:33<00:00, 11.33s/it]

    > [01:25:21.26] Baseline under DeepFool                    Acc=0.739 F1=0.325 Prec=0.358 Rec=0.298 Infer=0.0010ms
    > [01:25:21.26] DeepFool success rate / L_inf              24.2% / 0.0499


In [13]:
# ===== SECTION 12: Square Attack vs baseline =====
log.section("Square Attack vs baseline")

N_SQ = min(CONFIG["n_square_samples"], len(X_test))
idx_sq = np.random.RandomState(SEED).choice(len(X_test), N_SQ, replace=False)
X_sq, y_sq = X_test[idx_sq], y_test[idx_sq]

X_adv_square, square_success_before = tabular_square_attack(baseline_model, X_sq, y_sq, DEVICE, EPS, desc="Square (baseline)")
square_before = evaluate(baseline_model, X_adv_square, y_sq, DEVICE, "Baseline under Square")
square_before["success_rate"] = square_success_before



>>> [01:25:21.35] SECTION 10: Square Attack vs baseline


Square (baseline): 100%|██████████| 300/300 [00:07<00:00, 41.13it/s, remaining=584, total=1000]

    > [01:25:28.69] Baseline under Square                      Acc=0.584 F1=0.165 Prec=0.143 Rec=0.193 Infer=0.0020ms


In [14]:
# ===== SECTION 13: SHAP robust feature selection =====
log.section("SHAP feature stability analysis")

N_SHAP = min(CONFIG["n_shap_samples"], len(X_atk))
N_SHAP_BG = min(CONFIG["shap_background_samples"], len(X_train_bal))
background = X_train_bal[np.random.RandomState(SEED).choice(len(X_train_bal), N_SHAP_BG, replace=False)]

def predict_fn(x):
    return predict_proba_numpy(baseline_model, x, DEVICE)

explainer = shap.Explainer(predict_fn, background)
shap_clean = explainer(X_atk[:N_SHAP])
shap_adv = explainer(X_adv_store["PGD"][:N_SHAP])

shap_clean_vals = np.abs(shap_clean.values[..., 1]).mean(axis=0)
shap_adv_vals = np.abs(shap_adv.values[..., 1]).mean(axis=0)
stability = 1 - np.abs(shap_clean_vals - shap_adv_vals) / (shap_clean_vals + 1e-9)
stability_series = pd.Series(stability, index=selected_features).sort_values(ascending=False)

robust_features = stability_series.head(CONFIG["top_t_shap"]).index.tolist()
log.ok(f"Selected {len(robust_features)} SHAP-robust features: {robust_features}")



>>> [01:25:28.73] SECTION 11: SHAP feature stability analysis


PermutationExplainer explainer: 301it [00:23, 13.00it/s]                         
PermutationExplainer explainer: 301it [00:11,  3.89it/s]                         

    v [01:26:03.92] Selected 15 SHAP-robust features: ['Fwd Packet Length Max', 'Packet Length Std', 'Bwd Packet Length Mean', 'Avg Bwd Segment Size', 'Bwd Packet Length Max', 'Bwd Packet Length Std', 'Init Fwd Win Bytes', 'Packet Length Variance', 'Packet Length Mean', 'Avg Fwd Segment Size', 'Avg Packet Size', 'Fwd IAT Mean', 'Init Bwd Win Bytes', 'Packet Length Max', 'Fwd Packet Length Mean']


In [15]:
# ===== SECTION 14: defended model - Stage 1 (clean, robust features) =====
log.section("Defended model - Stage 1")

feat_idx = [selected_features.index(f) for f in robust_features]
X_train_def = X_train_bal[:, feat_idx]
X_test_def = X_test[:, feat_idx]
X_atk_def = X_atk[:, feat_idx]
mutable_mask_def = build_mutable_mask(robust_features, CONFIG["structural_features"])

defended_model = IDSNet(len(robust_features))
log.info(f"defended input_dim={len(robust_features)}")

t0 = time.time()
train_model(defended_model, X_train_def, y_train_bal, epochs=CONFIG["epochs_defended_stage1"],
            batch_size=CONFIG["batch_size"], device=DEVICE, desc="Stage 1")
t_stage1_time = time.time() - t0
log.ok(f"Stage 1 trained in {t_stage1_time/60:.2f} min")

torch.save(defended_model.state_dict(), f"{OUTPUT_DIR}/checkpoints/stage1_seed{SEED}.pt")



>>> [01:26:03.98] SECTION 12: Defended model - Stage 1
    . [01:26:04.53] defended input_dim=15


Stage 1: 100%|██████████| 50/50 [1:11:41<00:00, 86.02s/it, loss=0.1619, val_acc=0.9450]

    v [02:37:47.98] Stage 1 trained in 71.72 min


In [16]:
# ===== SECTION 15: adversarial training data generation =====
log.section("Adversarial training data generation")

clip_values_def = (float(X_train_def.min()), float(X_train_def.max()))
classifier_def, _ = make_art_classifier(defended_model, len(robust_features), clip_values_def)
attacks_def = make_attacks(classifier_def, EPS)

N_GEN = min(CONFIG["n_adv_train_samples"], len(X_train_def))
gen_idx = np.random.RandomState(SEED).choice(len(X_train_def), N_GEN, replace=False)
X_gen, y_gen = X_train_def[gen_idx], y_train_bal[gen_idx]

t0 = time.time()
adv_train_X, adv_train_y = [], []
for name, atk in tqdm(list(attacks_def.items()), desc="Adv training data"):
    X_adv_gen = atk.generate(x=X_gen, y=to_onehot(y_gen))
    X_adv_gen = apply_feature_mask(X_adv_gen, X_gen, mutable_mask_def)
    if name == "DeepFool":
        X_adv_gen = cap_to_eps(X_adv_gen, X_gen, EPS, clip_values_def)
    adv_train_X.append(X_adv_gen)
    adv_train_y.append(y_gen)
t_gen_time = time.time() - t0

X_adv_train = np.vstack(adv_train_X).astype(np.float32)
y_adv_train = np.concatenate(adv_train_y)
X_train_final = np.vstack([X_train_def, X_adv_train]).astype(np.float32)
y_train_final = np.concatenate([y_train_bal, y_adv_train])
log.ok(f"Combined train set: {X_train_final.shape}")



>>> [02:37:48.10] SECTION 13: Adversarial training data generation


Adv training data: 100%|██████████| 3/3 [01:58<00:00, 39.37s/it]


    v [02:39:46.83] Combined train set: (7631036, 15)


In [17]:
# ===== SECTION 16: defended model - Stage 2 (adversarial fine-tuning) =====
log.section("Defended model - Stage 2")

t0 = time.time()
train_model(defended_model, X_train_final, y_train_final, epochs=CONFIG["epochs_defended_stage2"],
            batch_size=CONFIG["batch_size"], device=DEVICE, desc="Stage 2")
t_stage2_time = time.time() - t0
log.ok(f"Stage 2 trained in {t_stage2_time/60:.2f} min")

defended_train_time_total = t_stage1_time + t_gen_time + t_stage2_time
torch.save(defended_model.state_dict(), f"{OUTPUT_DIR}/checkpoints/defended_seed{SEED}.pt")



>>> [02:39:46.91] SECTION 14: Defended model - Stage 2


Stage 2: 100%|██████████| 15/15 [23:59<00:00, 95.95s/it, loss=0.1622, val_acc=0.9452]

    v [03:03:48.47] Stage 2 trained in 24.03 min


In [18]:
# ===== SECTION 17: defended model evaluation =====
log.section("Defended model evaluation")

defended_clean_metrics = evaluate(defended_model, X_test_def, y_test, DEVICE, "Defended (clean)")

clip_values_def2 = (float(X_test_def.min()), float(X_test_def.max()))
classifier_def2, _ = make_art_classifier(defended_model, len(robust_features), clip_values_def2)
attacks_def2 = make_attacks(classifier_def2, EPS)

adv_results_after = {}
for name, atk in tqdm(list(attacks_def2.items()), desc="Attacks vs defended"):
    X_adv2 = atk.generate(x=X_atk_def, y=to_onehot(y_atk))
    X_adv2 = apply_feature_mask(X_adv2, X_atk_def, mutable_mask_def)
    if name == "DeepFool":
        X_adv2 = cap_to_eps(X_adv2, X_atk_def, EPS, clip_values_def2)
    adv_results_after[name] = evaluate(defended_model, X_adv2, y_atk, DEVICE, f"Defended under {name}")
    diag = attack_diagnostics(defended_model, X_atk_def, X_adv2, y_atk, DEVICE)
    adv_results_after[name].update(diag)
    log.metric(f"{name} success rate (defended)", f"{diag['success_rate']*100:.1f}%")

X_sq_def = X_sq[:, feat_idx]
X_adv_square2, square_success_after = tabular_square_attack(defended_model, X_sq_def, y_sq, DEVICE, EPS, desc="Square (defended)")
square_after = evaluate(defended_model, X_adv_square2, y_sq, DEVICE, "Defended under Square")
square_after["success_rate"] = square_success_after



>>> [03:03:48.56] SECTION 15: Defended model evaluation
    > [03:03:49.54] Defended (clean)                           Acc=0.959 F1=0.907 Prec=0.892 Rec=0.922 Infer=0.0005ms


Attacks vs defended:  33%|███▎      | 1/3 [00:00<00:00,  6.54it/s]

    > [03:03:49.70] Defended under FGSM                        Acc=0.929 F1=0.807 Prec=0.961 Rec=0.695 Infer=0.0010ms
    > [03:03:49.71] FGSM success rate (defended)               5.9%



Attacks vs defended:  67%|██████▋   | 2/3 [00:03<00:01,  1.82s/it]

    > [03:03:52.69] Defended under PGD                         Acc=0.771 F1=0.460 Prec=0.459 Rec=0.461 Infer=0.0010ms
    > [03:03:52.70] PGD success rate (defended)                19.6%



Attacks vs defended: 100%|██████████| 3/3 [02:09<00:00, 43.05s/it]


    > [03:05:58.70] Defended under DeepFool                    Acc=0.898 F1=0.740 Prec=0.803 Rec=0.686 Infer=0.0020ms
    > [03:05:58.70] DeepFool success rate (defended)           6.9%


Square (defended): 100%|██████████| 300/300 [00:08<00:00, 33.67it/s, remaining=737, total=1000]

    > [03:06:07.63] Defended under Square                      Acc=0.737 F1=0.320 Prec=0.354 Rec=0.292 Infer=0.0015ms


In [19]:
# ===== SECTION 18: summary tables =====
log.section("Summary tables")

def row(name, m, is_clean=False):
    sr = "-" if is_clean else f"{m.get('success_rate', float('nan'))*100:.2f}"
    return (name, round(m["accuracy"]*100, 2), round(m["f1"]*100, 2),
            round(m["precision"]*100, 2), round(m["recall"]*100, 2), sr)

rows = [row("Baseline (clean)", baseline_clean_metrics, True)]
for n in ["FGSM", "PGD", "DeepFool"]:
    rows.append(row(f"{n} (baseline)", adv_results_before[n]))
rows.append(row("Defended (clean)", defended_clean_metrics, True))
for n in ["FGSM", "PGD", "DeepFool"]:
    rows.append(row(f"{n} (defended)", adv_results_after[n]))

table4 = pd.DataFrame(rows, columns=["Condition", "Accuracy%", "F1%", "Precision%", "Recall%", "Success Rate%"])
print(table4.to_string(index=False))

table5 = pd.DataFrame([
    ("Baseline", square_before["accuracy"]*100, square_before["f1"]*100,
     square_before["precision"]*100, square_before["recall"]*100, square_before["success_rate"]*100),
    ("Defended", square_after["accuracy"]*100, square_after["f1"]*100,
     square_after["precision"]*100, square_after["recall"]*100, square_after["success_rate"]*100),
], columns=["Model", "Accuracy%", "F1%", "Precision%", "Recall%", "Success Rate%"]).round(2)
print()
print(table5.to_string(index=False))

overhead_retrain_pct = (t_gen_time + t_stage2_time) / baseline_train_time * 100
overhead_full_pct = defended_train_time_total / baseline_train_time * 100

table6 = pd.DataFrame([
    ("Baseline", baseline_train_time/3600, baseline_clean_metrics["infer_ms"], "-", "-"),
    ("Defended", defended_train_time_total/3600, defended_clean_metrics["infer_ms"],
     f"{overhead_retrain_pct:.1f}%", f"{overhead_full_pct:.1f}%"),
], columns=["Model", "Train hrs", "Infer ms/sample", "Overhead(retrain)", "Overhead(full)"])
print()
print(table6.to_string(index=False))



>>> [03:06:07.69] SECTION 16: Summary tables
          Condition  Accuracy%   F1%  Precision%  Recall% Success Rate%
   Baseline (clean)      95.40 89.74       87.29    92.34             -
    FGSM (baseline)      74.95 26.00       34.65    20.80         21.56
     PGD (baseline)      68.60 17.59       19.76    15.84         27.87
DeepFool (baseline)      73.85 32.52       35.80    29.79         24.19
   Defended (clean)      95.87 90.67       89.22    92.17             -
    FGSM (defended)      92.95 80.66       96.08    69.50          5.89
     PGD (defended)      77.10 45.99       45.88    46.10         19.65
DeepFool (defended)      89.80 73.98       80.33    68.56          6.93

   Model  Accuracy%   F1%  Precision%  Recall%  Success Rate%
Baseline       58.4 16.47       14.34    19.34          38.91
Defended       73.7 32.04       35.43    29.25          23.39

   Model  Train hrs  Infer ms/sample Overhead(retrain) Overhead(full)
Baseline   1.189696         0.000417            

In [20]:
# ===== SECTION 19: epsilon sweep (optional) =====
if CONFIG["run_eps_sweep"]:
    log.section("Epsilon sweep")
    sweep_rows = []
    for eps_val in tqdm(CONFIG["eps_sweep_values"], desc="Eps sweep"):
        atk_b = make_attacks(classifier, eps_val)
        atk_d = make_attacks(classifier_def2, eps_val)
        for name, atk in atk_b.items():
            Xs = apply_feature_mask(atk.generate(x=X_atk, y=to_onehot(y_atk)), X_atk, mutable_mask)
            if name == "DeepFool":
                Xs = cap_to_eps(Xs, X_atk, eps_val, clip_values)
            m = evaluate(baseline_model, Xs, y_atk, DEVICE)
            sweep_rows.append(("baseline", name, eps_val, m["accuracy"]))
        for name, atk in atk_d.items():
            Xs = apply_feature_mask(atk.generate(x=X_atk_def, y=to_onehot(y_atk)), X_atk_def, mutable_mask_def)
            if name == "DeepFool":
                Xs = cap_to_eps(Xs, X_atk_def, eps_val, clip_values_def2)
            m = evaluate(defended_model, Xs, y_atk, DEVICE)
            sweep_rows.append(("defended", name, eps_val, m["accuracy"]))

    eps_sweep_df = pd.DataFrame(sweep_rows, columns=["model", "attack", "eps", "accuracy"])
    eps_sweep_df.to_csv(f"{OUTPUT_DIR}/eps_sweep.csv", index=False)

    fig, ax = plt.subplots(figsize=(8, 5))
    for (m, a), grp in eps_sweep_df.groupby(["model", "attack"]):
        grp = grp.sort_values("eps")
        ax.plot(grp["eps"], grp["accuracy"]*100, marker="o", label=f"{m}-{a}")
    ax.set_xlabel("eps"); ax.set_ylabel("Accuracy %"); ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/eps_sweep_plot.png", dpi=150)
    plt.close()
    log.ok("Saved eps_sweep.csv + eps_sweep_plot.png")
else:
    log.info("Eps sweep skipped (CONFIG['run_eps_sweep']=False)")



>>> [03:06:07.72] SECTION 17: Epsilon sweep


PGD - Batches: 100%|██████████| 63/63 [00:02<00:00, 22.01it/s]
                                                              
DeepFool: 100%|██████████| 2000/2000 [00:32<00:00, 62.20it/s]

PGD - Batches: 100%|██████████| 63/63 [00:02<00:00, 23.40it/s]
                                                              
PGD - Batches: 100%|██████████| 63/63 [00:02<00:00, 22.71it/s]
                                                              
DeepFool: 100%|██████████| 2000/2000 [00:31<00:00, 62.50it/s]

PGD - Batches: 100%|██████████| 63/63 [00:02<00:00, 23.60it/s]
                                                              
PGD - Batches: 100%|██████████| 63/63 [00:02<00:00, 22.72it/s]
                                                              
DeepFool: 100%|██████████| 2000/2000 [00:32<00:00, 61.59it/s]

PGD - Batches: 100%|██████████| 63/63 [00:02<00:00, 23.77it/s]
                                                              
PGD - Batches: 100%|██████████| 63/63 [00:02<00:00, 22.

    v [03:18:18.05] Saved eps_sweep.csv + eps_sweep_plot.png


In [21]:
# ===== SECTION 20: save all artifacts =====
log.section("Saving artifacts")

all_metrics = {
    "baseline_clean": baseline_clean_metrics,
    "baseline_under_attack": adv_results_before,
    "baseline_under_square": square_before,
    "defended_clean": defended_clean_metrics,
    "defended_under_attack": adv_results_after,
    "defended_under_square": square_after,
}
with open(f"{OUTPUT_DIR}/all_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)

run_config = dict(CONFIG)
run_config.update({
    "device": str(DEVICE),
    "n_attack_samples_used": N_ATTACK,
    "n_square_samples_used": N_SQ,
    "n_shap_samples_used": N_SHAP,
    "n_adv_train_samples_used": N_GEN,
    "baseline_train_time_sec": baseline_train_time,
    "defended_stage1_time_sec": t_stage1_time,
    "defended_gen_time_sec": t_gen_time,
    "defended_stage2_time_sec": t_stage2_time,
    "defended_total_time_sec": defended_train_time_total,
    "overhead_retrain_only_pct": overhead_retrain_pct,
    "overhead_full_pipeline_pct": overhead_full_pct,
})
with open(f"{OUTPUT_DIR}/run_config.json", "w") as f:
    json.dump(run_config, f, indent=2)

table4.to_csv(f"{OUTPUT_DIR}/table4.csv", index=False)
table5.to_csv(f"{OUTPUT_DIR}/table5.csv", index=False)
table6.to_csv(f"{OUTPUT_DIR}/table6.csv", index=False)
mi_scores.to_csv(f"{OUTPUT_DIR}/mi_scores.csv")
stability_series.to_csv(f"{OUTPUT_DIR}/shap_stability_scores.csv")
with open(f"{OUTPUT_DIR}/selected_features.json", "w") as f:
    json.dump({"mi_features": selected_features, "shap_robust_features": robust_features}, f, indent=2)

torch.save(scaler, f"{OUTPUT_DIR}/scaler.pt")

log.ok(f"All artifacts saved to {OUTPUT_DIR}")



>>> [03:18:18.13] SECTION 18: Saving artifacts
    v [03:18:18.14] All artifacts saved to ./output/seed_42
